Imports

In [1]:
import pandas as pd
import numpy as np
from dep2pyodbc import dep2connection

pd.set_option("display.max_columns", None)

Connection to database

In [2]:
channel_crh = dep2connection("CRH")
channel_dwh_lisa = dep2connection("CRH_DWH")
cursor = channel_dwh_lisa.cursor()

Python-dotenv could not parse statement starting at line 22
Python-dotenv could not parse statement starting at line 22


pyodbc using windows
pyodbc using windows


Get decoded data from by using the controler script and drop unnecessary columns

In [3]:
df_decoded = pd.read_csv("./raw_data/decoded_fca.csv")
df_decoded.drop(columns=["Sequence1", "Sequence2", "Sequence3", "InstrumentClassId"], inplace=True)
df_decoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035545 entries, 0 to 3035544
Data columns (total 23 columns):
 #   Column                     Dtype
---  ------                     -----
 0   CandidateId                int64
 1   InstanceId                 int64
 2   ItemId                     int64
 3   TimeSpent                  int64
 4   Answer1                    int64
 5   Answer2                    int64
 6   Answer3                    int64
 7   Competency_1               int64
 8   Correct_answer_comp1_seq1  int64
 9   Correct_answer_comp1_seq2  int64
 10  Correct_answer_comp1_seq3  int64
 11  Competency_2               int64
 12  Correct_answer_comp2_seq1  int64
 13  Correct_answer_comp2_seq2  int64
 14  Correct_answer_comp2_seq3  int64
 15  Competency_3               int64
 16  Correct_answer_comp3_seq1  int64
 17  Correct_answer_comp3_seq2  int64
 18  Correct_answer_comp3_seq3  int64
 19  Competency_4               int64
 20  Correct_answer_comp4_seq1  int64
 21  Correct_

In [4]:
# df_decoded.head()

Get the other data from the FCA tests

In [5]:
df_not_coded = pd.read_sql("SELECT ID, CandidateID, InstanceID, TestStartTime, TestFinishTime, Data, CreatedDate, ModifiedDate, VersionNumber, PostedDate FROM CandidateResultFCA", channel_crh)

# df_not_coded.head()

C:\Users\verho\AppData\Local\Temp\ipykernel_87824\2525786402.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_not_coded = pd.read_sql("SELECT ID, CandidateID, InstanceID, TestStartTime, TestFinishTime, Data, CreatedDate, ModifiedDate, VersionNumber, PostedDate FROM CandidateResultFCA", channel_crh)


Combine decoded and other data from FCA in one dataframe

In [6]:
df_fca = pd.merge(df_decoded, df_not_coded, left_on=["CandidateId", "InstanceId"], right_on=["CandidateID", "InstanceID"], how="left")

# df_fca.head()

In [7]:
del df_decoded
del df_not_coded
# df_fca.info()

In [8]:
# df_fca[df_fca["CreatedDate"].isnull()]

Get the necesarry data from DimCandidate to use the correct keys in the FCA dataframe and drop columns that are no longer necessary

In [9]:
df_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)
# df_candidate.head()

C:\Users\verho\AppData\Local\Temp\ipykernel_87824\4219519558.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_candidate = pd.read_sql("SELECT CandidateKey, ID, InstanceID FROM DimCandidate", channel_dwh_lisa)


In [10]:
df_fca = pd.merge(df_fca, df_candidate, left_on=["CandidateId", "InstanceId"], right_on=["ID", "InstanceID"], how="left")
# df_fca.info()

In [11]:
# df_fca[["InstanceId", "InstanceID_x", "InstanceID_y"]].head()

In [12]:
df_fca.drop(columns=["CandidateId", "ID_y", "CandidateID", "InstanceId", "InstanceID_x", "InstanceID_y"], inplace=True)
df_fca.rename(columns={"ID_x": "TestID"}, inplace=True)
# df_fca.head()

In [13]:
del df_candidate
# df_fca.info()

Get the necesarry data from DimDate to use the correct keys in the FCA dataframe and drop columns that are no longer necessary

In [14]:
df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)
# df_dates['Date'] = pd.to_datetime(df_dates["Date"], format="%Y-%m-%d")
df_dates

C:\Users\verho\AppData\Local\Temp\ipykernel_87824\1357892697.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dates = pd.read_sql("SELECT DateKey, Date FROM DimDate", channel_dwh_lisa)


,DateKey,Date
0,20070101,2007-01-01
1,20070102,2007-01-02
2,20070103,2007-01-03
3,20070104,2007-01-04
4,20070105,2007-01-05
...,...,...
6570,20241227,2024-12-27
6571,20241228,2024-12-28
6572,20241229,2024-12-29
6573,20241230,2024-12-30


* CreatedDate

In [15]:
df_fca["CreatedDate"] = pd.to_datetime(df_fca["CreatedDate"]).dt.date

df_fca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035705 entries, 0 to 3035704
Data columns (total 30 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   ItemId                     int64         
 1   TimeSpent                  int64         
 2   Answer1                    int64         
 3   Answer2                    int64         
 4   Answer3                    int64         
 5   Competency_1               int64         
 6   Correct_answer_comp1_seq1  int64         
 7   Correct_answer_comp1_seq2  int64         
 8   Correct_answer_comp1_seq3  int64         
 9   Competency_2               int64         
 10  Correct_answer_comp2_seq1  int64         
 11  Correct_answer_comp2_seq2  int64         
 12  Correct_answer_comp2_seq3  int64         
 13  Competency_3               int64         
 14  Correct_answer_comp3_seq1  int64         
 15  Correct_answer_comp3_seq2  int64         
 16  Correct_answer_comp3_seq3  int64    

In [16]:
# df_fca[df_fca["CreatedDate"].isnull()].head()

In [17]:
df_fca = pd.merge(df_fca, df_dates, left_on="CreatedDate", right_on="Date", how="left")
df_fca.drop(columns=["Date", "CreatedDate"], inplace=True)
df_fca.rename(columns={"DateKey": "CreatedDateKey"}, inplace=True)

In [18]:
# df_fca.info()

* ModifiedDate

In [19]:
df_fca["ModifiedDate"] = pd.to_datetime(df_fca["ModifiedDate"]).dt.date

df_fca = pd.merge(df_fca, df_dates, left_on="ModifiedDate", right_on="Date", how="left")
df_fca.drop(columns=["Date", "ModifiedDate"], inplace=True)
df_fca.rename(columns={"DateKey": "ModifiedDateKey"}, inplace=True)

df_fca.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035705 entries, 0 to 3035704
Data columns (total 30 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   ItemId                     int64         
 1   TimeSpent                  int64         
 2   Answer1                    int64         
 3   Answer2                    int64         
 4   Answer3                    int64         
 5   Competency_1               int64         
 6   Correct_answer_comp1_seq1  int64         
 7   Correct_answer_comp1_seq2  int64         
 8   Correct_answer_comp1_seq3  int64         
 9   Competency_2               int64         
 10  Correct_answer_comp2_seq1  int64         
 11  Correct_answer_comp2_seq2  int64         
 12  Correct_answer_comp2_seq3  int64         
 13  Competency_3               int64         
 14  Correct_answer_comp3_seq1  int64         
 15  Correct_answer_comp3_seq2  int64         
 16  Correct_answer_comp3_seq3  int64    

* PostedDate

In [20]:
df_fca["PostedDate"] = pd.to_datetime(df_fca["PostedDate"]).dt.date

df_fca = pd.merge(df_fca, df_dates, left_on="PostedDate", right_on="Date", how="left")
df_fca.drop(columns=["Date", "PostedDate"], inplace=True)
df_fca.rename(columns={"DateKey": "PostedDateKey"}, inplace=True)

# df_fca.info()

In [21]:
del df_dates

Get the necesarry data from DimTime to use the correct keys in the FCA dataframe and drop columns that are no longer necessary

In [22]:
df_time = pd.read_sql("SELECT TimeKey, tekst FROM DimTime", channel_dwh_lisa)

df_time["tekst"] = pd.to_datetime(df_time["tekst"], format='%H:%M:%S').dt.time

# df_time.head()

C:\Users\verho\AppData\Local\Temp\ipykernel_87824\4115875712.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_time = pd.read_sql("SELECT TimeKey, tekst FROM DimTime", channel_dwh_lisa)


* TestStartTime

In [23]:
df_fca["TestStartTime"] = pd.to_datetime(df_fca["TestStartTime"]).dt.round('s').dt.time

df_fca = pd.merge(df_fca, df_time, left_on="TestStartTime", right_on="tekst", how="left")
df_fca.drop(columns=["tekst", "TestStartTime"], inplace=True)
df_fca.rename(columns={"TimeKey": "TestStartTimeKey"}, inplace=True)

# df_fca.info()

* TestFinishTime

In [24]:
df_fca["TestFinishTime"] = pd.to_datetime(df_fca["TestFinishTime"]).dt.round('s').dt.time

df_fca = pd.merge(df_fca, df_time, left_on="TestFinishTime", right_on="tekst", how="left")
df_fca.drop(columns=["tekst", "TestFinishTime"], inplace=True)
df_fca.rename(columns={"TimeKey": "TestFinishTimeKey"}, inplace=True)

# df_fca.info()

In [25]:
del df_time

Split dataftame in a test dataframe, a question dataframe and a competence dataframe

* Test dataframe

In [26]:
df_reserve = df_fca

In [27]:
df_test = df_fca[[
    "TestID", "CandidateKey", "CreatedDateKey", "ModifiedDateKey", "PostedDateKey", 
    "TestStartTimeKey", "TestFinishTimeKey", "VersionNumber"]]

In [28]:
columns_test = df_test.columns
print(columns_test)
columns_test = columns_test.drop("TestID")
print(columns_test)
columns_test = columns_test.drop("TestStartTimeKey")
print(columns_test)

Index(['TestID', 'CandidateKey', 'CreatedDateKey', 'ModifiedDateKey',
       'PostedDateKey', 'TestStartTimeKey', 'TestFinishTimeKey',
       'VersionNumber'],
      dtype='object')
Index(['CandidateKey', 'CreatedDateKey', 'ModifiedDateKey', 'PostedDateKey',
       'TestStartTimeKey', 'TestFinishTimeKey', 'VersionNumber'],
      dtype='object')
Index(['CandidateKey', 'CreatedDateKey', 'ModifiedDateKey', 'PostedDateKey',
       'TestFinishTimeKey', 'VersionNumber'],
      dtype='object')


In [29]:
df_test.drop_duplicates(inplace=True, subset=columns_test)
df_test.reset_index(inplace=True, drop=True)
df_test["TestKey"] = df_test.index + 1
# df_test.rename(columns={"Data" : "HexadecimaleString"}, inplace=True)

# df_test.head()

C:\Users\verho\AppData\Local\Temp\ipykernel_87824\792972555.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test.drop_duplicates(inplace=True, subset=columns_test)
C:\Users\verho\AppData\Local\Temp\ipykernel_87824\792972555.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["TestKey"] = df_test.index + 1


In [30]:
df_test = df_test[["TestKey", "TestID", "CandidateKey", "CreatedDateKey", "ModifiedDateKey", "PostedDateKey", "TestStartTimeKey", "TestFinishTimeKey", "VersionNumber"]]
# df_test.head()

In [31]:
columns_test

Index(['CandidateKey', 'CreatedDateKey', 'ModifiedDateKey', 'PostedDateKey',
       'TestFinishTimeKey', 'VersionNumber'],
      dtype='object')

In [32]:
df_test.value_counts("CandidateKey")

CandidateKey
50828301    1
50828221    1
50828141    1
50827891    1
50825221    1
           ..
854         1
654         1
544         1
284         1
114         1
Name: count, Length: 160064, dtype: int64

In [33]:
df_test[df_test["CandidateKey"] == 31371831]

,TestKey,TestID,CandidateKey,CreatedDateKey,ModifiedDateKey,PostedDateKey,TestStartTimeKey,TestFinishTimeKey,VersionNumber
54787,54788,9410,31371831,20151112,20151112,20191121,200854,205508,v1.0


* Competence dataframe

In [34]:
df_competence1 = df_fca[["Competency_1", "ItemId", "Correct_answer_comp1_seq1", "Correct_answer_comp1_seq2", "Correct_answer_comp1_seq3"]]
df_competence2 = df_fca[["Competency_2", "ItemId", "Correct_answer_comp2_seq1", "Correct_answer_comp2_seq2", "Correct_answer_comp2_seq3"]]
df_competence3 = df_fca[["Competency_3", "ItemId", "Correct_answer_comp3_seq1", "Correct_answer_comp3_seq2", "Correct_answer_comp3_seq3"]]
df_competence4 = df_fca[["Competency_4", "ItemId", "Correct_answer_comp4_seq1", "Correct_answer_comp4_seq2", "Correct_answer_comp4_seq3"]]
df_competence1.rename(columns={"Competency_1": "CompetenceCode", "Correct_answer_comp1_seq1":"IdealAnswerSequence1", "Correct_answer_comp1_seq2":"IdealAnswerSequence2", "Correct_answer_comp1_seq3":"IdealAnswerSequence3"}, inplace=True)
df_competence2.rename(columns={"Competency_2": "CompetenceCode", "Correct_answer_comp2_seq1":"IdealAnswerSequence1", "Correct_answer_comp2_seq2":"IdealAnswerSequence2", "Correct_answer_comp2_seq3":"IdealAnswerSequence3"}, inplace=True)
df_competence3.rename(columns={"Competency_3": "CompetenceCode", "Correct_answer_comp3_seq1":"IdealAnswerSequence1", "Correct_answer_comp3_seq2":"IdealAnswerSequence2", "Correct_answer_comp3_seq3":"IdealAnswerSequence3"}, inplace=True)
df_competence4.rename(columns={"Competency_4": "CompetenceCode", "Correct_answer_comp4_seq1":"IdealAnswerSequence1", "Correct_answer_comp4_seq2":"IdealAnswerSequence2", "Correct_answer_comp4_seq3":"IdealAnswerSequence3"}, inplace=True)

df_competence = pd.concat([df_competence1, df_competence2, df_competence3, df_competence4])
df_competence.drop_duplicates(inplace=True)
df_competence.reset_index(inplace=True, drop=True)

# df_competence.info()

C:\Users\verho\AppData\Local\Temp\ipykernel_87824\167052668.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_competence1.rename(columns={"Competency_1": "CompetenceCode", "Correct_answer_comp1_seq1":"IdealAnswerSequence1", "Correct_answer_comp1_seq2":"IdealAnswerSequence2", "Correct_answer_comp1_seq3":"IdealAnswerSequence3"}, inplace=True)
C:\Users\verho\AppData\Local\Temp\ipykernel_87824\167052668.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_competence2.rename(columns={"Competency_2": "CompetenceCode", "Correct_answer_comp2_seq1":"IdealAnswerSequence1", "Correct_answer_comp2_seq2":"IdealAnswerSequence2

In [35]:
df_competence = df_competence[df_competence["CompetenceCode"] != 0]
df_competence["CompetenceKey"] = df_competence.index + 1

# df_competence.head()

In [36]:
# df_competence.tail()

* Question dataframe

In [37]:
df_question = df_fca.drop(
    columns=["CandidateKey", "CreatedDateKey", "ModifiedDateKey", "PostedDateKey", 
             "TestStartTimeKey", "TestFinishTimeKey", "VersionNumber", "Data"
             ])
df_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3035705 entries, 0 to 3035704
Data columns (total 22 columns):
 #   Column                     Dtype
---  ------                     -----
 0   ItemId                     int64
 1   TimeSpent                  int64
 2   Answer1                    int64
 3   Answer2                    int64
 4   Answer3                    int64
 5   Competency_1               int64
 6   Correct_answer_comp1_seq1  int64
 7   Correct_answer_comp1_seq2  int64
 8   Correct_answer_comp1_seq3  int64
 9   Competency_2               int64
 10  Correct_answer_comp2_seq1  int64
 11  Correct_answer_comp2_seq2  int64
 12  Correct_answer_comp2_seq3  int64
 13  Competency_3               int64
 14  Correct_answer_comp3_seq1  int64
 15  Correct_answer_comp3_seq2  int64
 16  Correct_answer_comp3_seq3  int64
 17  Competency_4               int64
 18  Correct_answer_comp4_seq1  int64
 19  Correct_answer_comp4_seq2  int64
 20  Correct_answer_comp4_seq3  int64
 21  TestID  

In [38]:
# df_question.head()

In [39]:
df_competence[(df_competence["CompetenceCode"] == 10) & (df_competence["ItemId"] == 422)]

,CompetenceCode,ItemId,IdealAnswerSequence1,IdealAnswerSequence2,IdealAnswerSequence3,CompetenceKey
20,10,422,1,4,5,21
130,10,422,1,2,4,131
180,10,422,1,5,2,181
236,10,422,2,4,3,237
256,10,422,1,2,5,257
332,10,422,3,5,1,333
385,10,422,2,3,4,386
473,10,422,4,2,3,474


In [40]:
for i in [1,2,3,4]:
    df_question = pd.merge(left=df_question, right=df_competence, how='left',
                        left_on=["ItemId", f"Competency_{i}", f"Correct_answer_comp{i}_seq1", f"Correct_answer_comp{i}_seq2", f"Correct_answer_comp{i}_seq3"],
                        right_on=["ItemId", "CompetenceCode", "IdealAnswerSequence1", "IdealAnswerSequence2", "IdealAnswerSequence3"])
    df_question.drop(columns=[f"Competency_{i}", "CompetenceCode", "IdealAnswerSequence1", "IdealAnswerSequence2", "IdealAnswerSequence3"], inplace=True)
    df_question.rename(columns={"CompetenceKey": f"Competence{i}Key"}, inplace=True)

df_question.rename(columns={
    "Answer1" : "AnswerSequence1",
    "Answer2" : "AnswerSequence2",
    "Answer3" : "AnswerSequence3",
})
df_question = df_question.drop(columns=["Correct_answer_comp1_seq1",
                                   "Correct_answer_comp1_seq2",
                                   "Correct_answer_comp1_seq3",
                                   "Correct_answer_comp2_seq1",
                                   "Correct_answer_comp2_seq2",
                                   "Correct_answer_comp2_seq3",
                                   "Correct_answer_comp3_seq1",
                                   "Correct_answer_comp3_seq2",
                                   "Correct_answer_comp3_seq3",
                                   "Correct_answer_comp4_seq1",
                                   "Correct_answer_comp4_seq2",
                                   "Correct_answer_comp4_seq3",])
df_question["QuestionKey"] = df_question.index + 1
df_question.tail()

,ItemId,TimeSpent,Answer1,Answer2,Answer3,TestID,Competence1Key,Competence2Key,Competence3Key,Competence4Key,QuestionKey
3035700,422,80,4,5,2,119246,68,541.0,NaN,NaN,3035701
3035701,423,98,5,3,4,119246,69,NaN,NaN,NaN,3035702
3035702,424,126,1,2,5,119246,70,NaN,NaN,NaN,3035703
3035703,425,443,5,4,1,119246,71,NaN,NaN,NaN,3035704
3035704,426,37,3,5,2,119246,72,NaN,NaN,NaN,3035705


In [41]:
# df_question.info()

In [42]:
df_question = df_question.replace(np.nan, 0)
df_question["Competence2Key"] = df_question["Competence2Key"].astype(int)
df_question["Competence3Key"] = df_question["Competence3Key"].astype(int)
df_question["Competence4Key"] = df_question["Competence4Key"].astype(int)

# df_question.head()

In [43]:
df_test

,TestKey,TestID,CandidateKey,CreatedDateKey,ModifiedDateKey,PostedDateKey,TestStartTimeKey,TestFinishTimeKey,VersionNumber
0,1,133595,544,20220908,20220908,20220908,93154,93212,V1.0
1,2,133598,854,20221207,20221207,20221207,134516,141829,v1.0
2,3,133604,7564,20230809,20230809,20230809,100349,103238,v1.0
3,4,133645,13914,20231103,20231103,20231103,95141,104332,v1.0
4,5,133637,14244,20231025,20231025,20231025,62301,71219,v1.0
...,...,...,...,...,...,...,...,...,...
160059,160060,120672,49206841,20231031,20231031,20231031,104057,110911,v1.0
160060,160061,120001,49208481,20231027,20231027,20231027,123533,130252,v1.0
160061,160062,121795,49209181,20231115,20231115,20231115,73708,80636,v1.0
160062,160063,121339,49210931,20231114,20231114,20231114,73921,80421,v1.0


In [44]:
df_question = pd.merge(df_question, df_test[["TestKey", "TestID"]], on="TestID", how="left")

# df_question.info()

In [45]:
df_test.drop(columns=["TestID"], inplace=True)
df_question.drop(columns=["TestID"], inplace=True)

In [46]:
df_question.head()

,ItemId,TimeSpent,Answer1,Answer2,Answer3,Competence1Key,Competence2Key,Competence3Key,Competence4Key,QuestionKey,TestKey
0,222,11,0,0,0,1,0,0,0,1,1.0
1,223,3,3,4,2,2,183,0,0,2,1.0
2,226,13,4,3,2,3,184,0,0,3,1.0
3,229,100,2,0,4,4,0,0,0,4,1.0
4,238,2,0,0,0,5,0,0,0,5,1.0


In [47]:
df_question = df_question[[
    "QuestionKey", "TestKey", 
    "Competence1Key", "Competence2Key", "Competence3Key", "Competence4Key", 
    "ItemId", "Answer1", "Answer2", "Answer3", "TimeSpent"
    ]]

In [44]:
# df_fca = df_fca[[
#        "TestID",
#        "Data", 
#        "VersionNumber", 
#        "CandidateKey",
#        "CreatedDateKey",
#        "ModifiedDateKey",
#        "PostedDateKey",
#        "TestStartTimeKey",
#        "TestFinishTimeKey"]]

# df_fca.drop_duplicates(inplace=True)
# df_fca.reset_index(inplace=True, drop=True)
# df_fca["FCATestKey"] = df_fca.index + 1
# df_fca.rename(columns={"Data" : "HexadecimaleString"}, inplace=True)

# df_fca

In [45]:
# fca_cols = df_fca.columns.to_list()
# fca_cols.remove("TestID")
# fca_cols.remove("FCATestKey")
# print(fca_cols)
# pd.merge(df_question, df_fca, on=["TestID"], how="inner").drop(columns=fca_cols)

In [46]:
# df_fca.drop(columns=["TestID"], inplace=True)

Result + to database:

* DimCompetence

In [47]:
df_competence = df_competence[["CompetenceKey", "CompetenceCode", "ItemId", "IdealAnswerSequence1", "IdealAnswerSequence2", "IdealAnswerSequence3"]]

# so SQL Server doesn't complain that there is a non existing foreign key
zeros = {"CompetenceKey": 0, "CompetenceCode": 0, "ItemId": 0, "IdealAnswerSequence1": 0, "IdealAnswerSequence2": 0, "IdealAnswerSequence3": 0}
df_competence = df_competence._append(zeros, ignore_index=True)


# df_competence.head()

In [48]:
# df_competence.info()

In [49]:
df_competence.to_csv('../decoded_data/FCA/DimCompetence.csv', index=False)

* FactTest

In [50]:
df_test['Test'] = 'FCA'

In [51]:
# df_test.head()

In [52]:
# df_test.info()

In [53]:
df_test.to_csv('../decoded_data/FCA/FactTest.csv', index=False)

* FactQuestionFCA

In [54]:
# df_question.head()

In [55]:
# df_question.info()

In [56]:
df_question.to_csv('../decoded_data/FCA/FactQuestionFCA.csv', index=False)